In [8]:
# 1. Importação de bibliotecas
## Importação das bibliotecas necessárias para executar o pipeline.

###Manipulação de caminhos e arquivos
from pathlib import Path

###Extração de dados do PDF
import pymupdf

###Manipulação de dados e leitura do Excel
import pandas as pd

###Conexão e execução de comandos SQL
import pyodbc

# Importação do módulo para identificação de padrões em textos
import re

In [5]:
# 2. Configuração dos caminhos
## Definição dos caminhos do PDF e do Excel utilizados como fontes de dados.

### Definição da pasta principal do projeto
pasta_projeto = Path.cwd().parent

### Caminho do demonstrativo financeiro em PDF
caminho_pdf = pasta_projeto / "data" / "raw" / "demonstracoes_financeiras" / "DF_CONSOLIDADA_WEG_ORIGINAL.pdf"

### Caminho do plano de contas em Excel
caminho_excel = pasta_projeto / "data" / "raw" / "plano_de_contas" / "DF_WEG.xlsx"

In [6]:
# 3. Leitura e Extração dos Dados do PDF
## Leitura da DRE consolidada e extração dos dados financeiros de 2023 a 2025.
### Abrir o demonstrativo financeiro
documento = pymupdf.open(caminho_pdf)

### Selecionar a página da DRE consolidada
pagina_dre = documento[17]

### Extrair o texto da página
texto_dre = pagina_dre.get_text()

### Visualizar o conteúdo extraído
print(texto_dre)

DFs Consolidadas / Demonstração do Resultado
(Reais Mil)
Código da 
Conta
Descrição da Conta
Último Exercício
01/01/2025 à 31/12/2025
Penúltimo Exercício
01/01/2024 à 31/12/2024
Antepenúltimo Exercício
01/01/2023 à 31/12/2023
3.01
Receita de Venda de Bens e/ou Serviços
40.804.110
37.986.941
32.503.601
3.02
Custo dos Bens e/ou Serviços Vendidos
-27.122.477
-25.173.096
-21.702.737
3.03
Resultado Bruto
13.681.633
12.813.845
10.800.864
3.04
Despesas/Receitas Operacionais
-5.682.892
-5.123.317
-4.338.739
3.04.01
Despesas com Vendas
-3.353.431
-2.987.307
-2.426.459
3.04.01.01
(Provisão) reversão ao valor recuperável de clientes
-74.335
-22.896
-17.939
3.04.01.02
Outras Despesas de Vendas
-3.279.096
-2.964.411
-2.408.520
3.04.02
Despesas Gerais e Administrativas
-1.513.882
-1.299.421
-1.044.888
3.04.02.01
Honorários dos Administradores
-21.427
-21.646
-21.368
3.04.02.02
Outras
-1.492.455
-1.277.775
-1.023.520
3.04.04
Outras Receitas Operacionais
222.777
163.792
155.466
3.04.05
Outras Despesas

In [13]:
# 4. Estruturação dos Dados da DRE

## Organização das contas contábeis e dos valores em um DataFrame.

### Separar o texto extraído do PDF em linhas.
linhas_dre = texto_dre.splitlines()

### Definir o padrão para identificar os códigos contábeis.
### Exemplos: 3.01, 3.04.02 e 3.04.02.01.
padrao_codigo = r"^3(?:\.\d+)+$"

### Definir o padrão dos valores financeiros apresentados no PDF.
### Exemplos: 40.804.110 e -27.122.477.
padrao_valor = r"^-?\d[\d.]*$"

### Identificar os códigos contábeis presentes no texto.
codigos_encontrados = [
    linha.strip()
    for linha in linhas_dre
    if re.fullmatch(padrao_codigo, linha.strip())
]

### Criar uma lista para armazenar as contas e seus valores.
registros_dre = []

### Percorrer as linhas e localizar cada conta contábil.
for indice, linha in enumerate(linhas_dre):
    codigo = linha.strip()

    if codigo in codigos_encontrados:

        #### Identificar a descrição e os valores dos três exercícios.
        descricao = linhas_dre[indice + 1].strip()
        valores = [
            item.strip()
            for item in linhas_dre[indice + 2:indice + 5]
        ]

        #### Incluir apenas os registros com três valores válidos.
        if len(valores) == 3 and all(
            re.fullmatch(padrao_valor, valor)
            for valor in valores
        ):

            ##### Converter os valores de milhares de reais para reais.
            registros_dre.append({
                "codigo": codigo,
                "descricao": descricao,
                "2025": int(valores[0].replace(".", "")) * 1000,
                "2024": int(valores[1].replace(".", "")) * 1000,
                "2023": int(valores[2].replace(".", "")) * 1000
            })

### Organizar os registros em um DataFrame.
df_dre = pd.DataFrame(registros_dre)

### Conferir a quantidade de contas identificadas e estruturadas.
print(f"Códigos identificados: {len(codigos_encontrados)}")
print(f"Contas estruturadas: {len(df_dre)}")

### Visualizar os dados organizados.
display(df_dre)

Códigos identificados: 28
Contas estruturadas: 25


,codigo,descricao,2025,2024,2023
0,3.01,Receita de Venda de Bens e/ou Serviços,40804110000,37986941000,32503601000
1,3.02,Custo dos Bens e/ou Serviços Vendidos,-27122477000,-25173096000,-21702737000
2,3.03,Resultado Bruto,13681633000,12813845000,10800864000
3,3.04,Despesas/Receitas Operacionais,-5682892000,-5123317000,-4338739000
4,3.04.01,Despesas com Vendas,-3353431000,-2987307000,-2426459000
5,3.04.01.01,(Provisão) reversão ao valor recuperável de cl...,-74335000,-22896000,-17939000
6,3.04.01.02,Outras Despesas de Vendas,-3279096000,-2964411000,-2408520000
7,3.04.02,Despesas Gerais e Administrativas,-1513882000,-1299421000,-1044888000
8,3.04.02.01,Honorários dos Administradores,-21427000,-21646000,-21368000
9,3.04.02.02,Outras,-1492455000,-1277775000,-1023520000


In [16]:
# 5. Leitura do Plano de Contas (Excel)
## Importação e inspeção das informações contábeis do arquivo Excel.

### Ler a aba PlanoContas e preservar os códigos contábeis como texto.
df_plano_contas = pd.read_excel(
    caminho_excel,
    sheet_name="PlanoContas",
    dtype={"Id Conta": str}
)

In [17]:
# 6. Conexão com o SQL Server
## Estabelecimento da conexão entre Python e SQL Server.

### Definir os parâmetros de conexão.
servidor = "localhost"
driver = "ODBC Driver 18 for SQL Server"

### Construir a string de conexão utilizando autenticação do Windows.
string_conexao = (
    f"DRIVER={{{driver}}};"
    f"SERVER={servidor};"
    "DATABASE=master;"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

### Estabelecer e testar a conexão com o SQL Server.
conexao = pyodbc.connect(string_conexao)

print("Conexão com o SQL Server estabelecida com sucesso!")

### Encerrar a conexão após o teste.
conexao.close()

Conexão com o SQL Server estabelecida com sucesso!


In [ ]:
#(Passo 7 em SQL)
# 7. Carregamento dos dados no SQL Server
## Conexão com o banco de dados DRE_WEG

string_conexao_dre = (
    f"DRIVER={{{driver}}};"
    f"SERVER={servidor};"
    "DATABASE=DRE_WEG;"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

conexao_dre = pyodbc.connect(string_conexao_dre)

print("Conexão com o banco DRE_WEG estabelecida com sucesso!")

### Conferir a quantidade de registros dos DataFrames
print("Registros da DRE:", len(df_dre))
print("Registros do Plano de Contas:", len(df_plano_contas))

### Conferir a quantidade de registros nas tabelas de Staging
cursor = conexao_dre.cursor()

for tabela in ["stg_DRE", "stg_PlanoContas"]:
    cursor.execute(f"SELECT COUNT(*) FROM dbo.{tabela}")
    quantidade = cursor.fetchone()[0]
    print(f"Registros em {tabela}: {quantidade}")

### Carregamento dos DataFrames nas tabelas de Staging

sql_dre = """
INSERT INTO dbo.stg_DRE
(codigo, [descrição], [2025], [2024], [2023])
VALUES (?, ?, ?, ?, ?)
"""

sql_plano = """
INSERT INTO dbo.stg_PlanoContas
([Id Conta], [Descrição], [Lançamento], [Calculado])
VALUES (?, ?, ?, ?)
"""

### Preparar os registros para inserção
dados_dre = list(
    df_dre[["codigo", "descricao", "2025", "2024", "2023"]]
    .itertuples(index=False, name=None)
)

dados_plano = list(
    df_plano_contas[
        ["Id Conta", "Descrição", "Lançamento", "Calculado"]
    ].itertuples(index=False, name=None)
)

### Substituir os dados anteriores e carregar os registros atuais
try:
    cursor.execute("DELETE FROM dbo.stg_DRE")
    cursor.execute("DELETE FROM dbo.stg_PlanoContas")

    cursor.executemany(sql_dre, dados_dre)
    cursor.executemany(sql_plano, dados_plano)

    # Conferir as quantidades antes de confirmar a transação
    cursor.execute("SELECT COUNT(*) FROM dbo.stg_DRE")
    total_dre = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM dbo.stg_PlanoContas")
    total_plano = cursor.fetchone()[0]

    if total_dre != len(dados_dre) or total_plano != len(dados_plano):
        raise ValueError("Divergência na quantidade de registros carregados.")

    conexao_dre.commit()

    print(f"Registros carregados em stg_DRE: {total_dre}")
    print(f"Registros carregados em stg_PlanoContas: {total_plano}")
    print("Carregamento concluído com sucesso!")

except Exception:
    conexao_dre.rollback()
    print("Erro no carregamento. Transação desfeita.")
    raise

finally:
    cursor.close()
    conexao_dre.close()

#(Passos Finais 9 e 10 em SQL, Passo 11 no Power BI)

Conexão com o banco DRE_WEG estabelecida com sucesso!
Registros da DRE: 25
Registros do Plano de Contas: 34
Registros em stg_DRE: 0
Registros em stg_PlanoContas: 0
Registros carregados em stg_DRE: 25
Registros carregados em stg_PlanoContas: 34
Carregamento concluído com sucesso!
